# Huni ground truth — note-level, from your own stems

## What changed, and why

The four ground truths produced by the previous version of this notebook
were structurally sound — all four fit inside their audio, no overlaps,
plausible note ranges. But measuring them exposed one serious defect:

| song | multi-row words | exactly equal slices |
|---|---|---|
| Araw-Araw - Ben&Ben | 59 | 59 (100%) |
| Dadalhin - Regine Velasquez | 104 | 104 (100%) |
| Fallen - Lola Amour | 51 | 51 (100%) |
| Kumpas - Moira Dela Torre | 95 | 95 (100%) |

**Every** within-word note boundary was arithmetic: a Whisper word span
divided by its vowel count. `Umagan` came out as 0.0 → 0.507 → 1.013 →
1.52, exactly thirds. The pitches were real; the moment a note changed
was invented.

It was wrong in both directions at once. About half of all rows sat in
same-pitch runs of up to 7, so held notes were chopped into pieces:

| song | rows | real note events |
|---|---|---|
| Araw-Araw | 198 | 134 |
| Dadalhin | 365 | 246 |
| Fallen | 247 | 173 |
| Kumpas | 300 | 216 |

## The unit is the note event

Not the line, not the word, not the syllable — the **note**, which is
what a transcriber writes on a staff. The mapping to lyrics is
deliberately not one-to-one:

- usually one syllable is one note
- one syllable can carry several notes (melisma — the measured data has
  7–20% of within-syllable transitions jumping 5+ semitones)
- several syllables can sit on the same pitch, as separate note events

So boundaries come from **pitch onsets**, and the lyric text is attached
afterwards as a label. It is never a target: the app scores notes only.

This also fixes the score. The app creates one scored segment per
ground-truth row, so row granularity *is* scoring granularity. Today a
note held across 5 rows counts five times, and a melisma collapsed into
one row compares the singer against a median of pitches nobody meant to
sing. Note-level rows make the score mean "how many notes did you hit".

No app change is needed: `syllable` is only carried and displayed, and
segments are grouped by time and by recognized word, never by syllable.

## The labels come from the lyrics now, not from Whisper

The first extraction run produced notes that passed every check — 504
notes inside a tight A3–C#4 register, 3% equal slices, no overshoot — and
word labels that did not. Scored against the real lyrics:

| | Nasa Iyo Na Ang Lahat, Whisper large-v3 |
|---|---|
| labels that are not words in the song | 39 of 483 (8%) |
| sequence error rate (upper bound) | 35% |
| substitutions | 38 |
| deletions | 58 |

Concretely: `pinili` → `binili`, `nirereto` → `nilireto`, `sarap` never
heard at all, `'yo na ang` swallowed into one 1.06 s span, and eight
notes labelled with a music-note emoji.

None of that is fixable by a bigger model, because free transcription is
answering the wrong question. It asks *what words are these*. We already
know what the words are. The question worth asking is *when is each of
these known words sung* — which is *forced alignment*, and it makes three
of those four failure modes impossible rather than less likely:
spellings and word order come from `backend/lyrics/`, so only the timing
is estimated.

Cell 6 does it with `MMS_FA`, a character-level CTC model from Meta's
Massively Multilingual Speech work that covers Tagalog. Because it
returns a span per *character*, a word can be cut at its syllable
boundaries — so a melisma held over four notes reads `la-a-a-hat`
instead of the word `lahat` four times, and the `syllable` field in the
exported JSON finally holds a syllable.

Whisper stays in the notebook for any song with no lyrics sheet, and
nothing downloads it unless a song needs it.

## What you need

Per song, two files from the **same separation run**:
- the **vocals** stem
- the **instrumental** stem (what the app will play)

They are sample-aligned with each other, so ground truth taken from the
vocals is exactly aligned to the instrumental. Nothing is estimated —
no Demucs, no DTW.

**MP3 is fine.** Encoder delay is deterministic, so two stems encoded the
same way shift by the same amount and stay aligned with each other, which
is all this depends on. You will ship MP3 regardless:
`SongAudioService` hardcodes `assets/audio/<Title - Artist>.mp3`. Use WAV
for the vocals if you have it, purely because lossy compression can
perturb pitch tracking at low bitrates; above ~192 kbps it makes no
practical difference.

What breaks alignment is **editing**, not format. If either stem is
trimmed, faded or normalised on its own after separation, the premise is
void. Cell 3 checks their durations agree.

Set **Runtime → Change runtime type → T4 GPU**.

## Cell 1 — install

In [ ]:
# One install per line, so a failure names the package that broke
# instead of taking the others down with it.
!pip install -q torchcrepe
!pip install -q openai-whisper
!pip install -q librosa soundfile mutagen

# torchaudio ships with Colab's torch, but pin nothing here: the
# forced aligner's weights are fetched by torchaudio itself, so a
# mismatched pip install is the one thing that breaks it.
import torchaudio
print('torchaudio', torchaudio.__version__)

# No demucs: your stems are already separated.
# No tensorflow: torchcrepe is the PyTorch port of CREPE, and TF would
# only fight Torch over the GPU.

import torch
print('torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE -- set Runtime > Change runtime type > T4 GPU')

## Cell 2 — upload the stems

Run twice: `WHICH = 'vocals'`, then `WHICH = 'instrumental'`.

**Name both stems of a pair identically**, using the app's convention:
`Kumpas - Moira Dela Torre.mp3`. Pairing is by filename. Note the spaces
around the dash — `Dadalhin- Regine Velasquez.mp3` does not resolve, and
that exact typo has already happened once.

In [ ]:
import os
import shutil
from google.colab import files

VOCALS_DIR = '/content/vocals'
INSTR_DIR = '/content/instrumental'
os.makedirs(VOCALS_DIR, exist_ok=True)
os.makedirs(INSTR_DIR, exist_ok=True)

WHICH = 'vocals'   # <<< 'vocals', run; then 'instrumental', run again

dest = VOCALS_DIR if WHICH == 'vocals' else INSTR_DIR
print(f'Select the {WHICH.upper()} stems...')
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join(dest, name))
    print(f'  -> {dest}/{name}')

print(f'\nvocals/      : {sorted(os.listdir(VOCALS_DIR))}')
print(f'instrumental/: {sorted(os.listdir(INSTR_DIR))}')

## Cell 3 — pair them up, and check they really are aligned

Reports only. Two stems from one separation run are the same length to
within a frame; more than a second apart means they did **not** come from
the same source and the premise is void for that song.

In [ ]:
import os
from mutagen import File as MutagenFile

VOCALS_DIR = '/content/vocals'
INSTR_DIR = '/content/instrumental'
AUDIO_EXT = ('.wav', '.mp3', '.flac', '.m4a', '.ogg')


def duration_of(path):
    info = MutagenFile(path)
    if info is None or not getattr(info, 'info', None):
        return None
    return float(info.info.length)


def listing(folder):
    if not os.path.isdir(folder):
        return {}
    return {os.path.splitext(f)[0].strip(): os.path.join(folder, f)
            for f in sorted(os.listdir(folder))
            if f.lower().endswith(AUDIO_EXT)}


vocals = listing(VOCALS_DIR)
instr = listing(INSTR_DIR)

PAIRS = {}
print(f"{'song':<38} {'vocals':>8} {'instr':>8} {'diff':>7}  status")
print('-' * 80)
for stem in sorted(set(vocals) | set(instr)):
    v, i = vocals.get(stem), instr.get(stem)
    if not v:
        print(f'{stem[:38]:<38} {"MISSING":>8} {"ok":>8} {"":>7}  '
              'no vocals stem')
        continue
    if not i:
        print(f'{stem[:38]:<38} {"ok":>8} {"MISSING":>8} {"":>7}  '
              'no instrumental stem')
        continue
    dv, di = duration_of(v), duration_of(i)
    if dv is None or di is None:
        print(f'{stem[:38]:<38} {"?":>8} {"?":>8} {"":>7}  '
              'unreadable duration')
        continue
    diff = di - dv
    if abs(diff) > 1.0:
        status = 'MISMATCH -- not from the same source, skipping'
    else:
        status = 'aligned'
        PAIRS[stem] = (v, i)
    print(f'{stem[:38]:<38} {dv:8.1f} {di:8.1f} {diff:+7.2f}  {status}')

print('-' * 80)
print(f'{len(PAIRS)} song(s) will be processed.')
if not PAIRS:
    print('\nNothing to do. Stems pair by filename, so both sides must')
    print('match exactly apart from the folder.')

## Cell 4 — settings

Self-contained. Re-run this alone if a later cell says a name is not
defined.

The note-segmentation numbers are the ones worth understanding:
`SEMITONE_TOL` is how far pitch may wander before it counts as a new
note (0.8 keeps vibrato as one note), and `HOLD_MS` is how long that
deviation must persist before it is believed (which is what stops a slide
from spawning a note per frame).

In [ ]:
HOP_MS = 10             # pitch hop; CREPE's own default
CONF_MIN = 0.50         # torchcrepe periodicity below this is not pitch

# Per song, because 0.50 is too strict for some stems. Buwan's notes
# covered 49s of a 272s span at 0.50 while forced alignment placed words
# across the whole song, so the vocal was there and the threshold was
# rejecting it.
CONF_MIN_BY_SONG = {
    # 'Buwan - Juan Karlos': 0.35,
}

# When a song still comes out under-covered, re-segment it once at
# RELAXED_CONF_MIN and keep whichever covered more. Costs seconds, not
# another run: the pitch track is already computed and only the threshold
# applied to it changes.
AUTO_RELAX_CONF = True
RELAXED_CONF_MIN = 0.35
MIN_COVERAGE_PCT = 45


def conf_min_for(stem):
    return CONF_MIN_BY_SONG.get(stem, CONF_MIN)
# None sizes these from the GPU that was actually allocated (see
# tuned_for_gpu below); set a number to override. Lower them on OOM.
BATCH_SIZE = None       # torchcrepe frames per forward pass

# --- the F0 search range ---------------------------------------------
#
# The floor matters more than it looks. Below the lowest note actually
# sung, a pitch tracker can lock onto the subharmonic and report half the
# true frequency, which lands exactly 12 semitones low -- inside any
# singable range, so every other check in Cell 9 passes it. Araw-Araw came
# out with 19 such notes, 9 of them on the same syllable.
#
# Set this just under the lowest note the singer really reaches. Cell 9
# prints the measured register so the next run can be checked against it.
#  150 Hz = D3, a male pop baritone's floor (Araw-Araw, Nasa Iyo)
#   98 Hz = G2, only for a genuinely low voice
#  175 Hz = F3, a typical female pop range
DEFAULT_FMIN_HZ = 150.0
# The ceiling matters for the same reason the floor does, mirrored: a
# tracker that doubles reports an octave too HIGH. A ten-song run topped
# out at A#5-C6 on several songs while the highest note these singers
# demonstrably reach is around G5, so 1100 Hz was leaving room for it.
# 800 Hz still allows G5 (784). Raise it per song for a real soprano.
DEFAULT_FMAX_HZ = 800.0
FMAX_BY_SONG = {
    # 'Dadalhin - Regine Velasquez': 1100.0,
}


def fmax_for(stem):
    return FMAX_BY_SONG.get(stem, DEFAULT_FMAX_HZ)


FMAX_HZ = DEFAULT_FMAX_HZ

# Per song, where the default clips something real or lets one through.
FMIN_BY_SONG = {
    # 'Mundo - IV Of Spades': 120.0,
}


def fmin_for(stem):
    return FMIN_BY_SONG.get(stem, DEFAULT_FMIN_HZ)        # torchcrepe frames per forward pass; lower on OOM

# --- note segmentation ---
SEMITONE_TOL = 0.8      # wander allowed within one note (vibrato is ~0.5)
HOLD_MS = 60            # a deviation must last this long to start a note
GAP_MS = 100            # a voicing gap this long ends a note (breath/rest)
MIN_NOTE_MS = 100       # shorter than this is an artifact, not a note
SMOOTH_MS = 50          # median filter width; kills jitter, keeps onsets
MIDI_LO, MIDI_HI = 36, 88   # plausible for a voice; octave errors land out

# --- word labels ---------------------------------------------------
#
# Note boundaries come from the pitch contour, so a wrong word costs a
# label and never a target. But the labels are how a human checks that a
# ground-truth file lines up with the recording, so they are worth
# getting right.
#
# Two ways to get them, and they are not close:
#
#   'align'    forced alignment of the real lyrics (Cell 6). Asks WHERE
#              each known word is, so spelling and word order come from
#              backend/lyrics/ and cannot be wrong. Only timing is
#              estimated. Needs an entry in LYRICS_BY_SONG.
#   'whisper'  free transcription. Asks WHAT the words are, which is a
#              far harder question for singing. Measured at 27% WER on
#              Nasa Iyo Na Ang Lahat with large-v3: it wrote `binili`
#              for `pinili`, dropped `sarap` entirely, swallowed
#              "'yo na ang" into one 1.06s span, and labelled 8 notes
#              with a music-note emoji.
#
# 'auto' aligns every song that has lyrics and falls back to Whisper for
# any that does not, which is the only reason to keep Whisper around.
# DEFAULT IS 'none', because nothing scores them. Turn labels on when a
# file passes every check in Cell 9 and still sounds offset: a constant
# time shift that still fits inside the audio is invisible to all of
# them, and reading the word under a note is the only way to see it.
# 'align' is also what produces the word onsets that time the app's
# scrolling lyrics (backend/sync_lyrics.py).
LABEL_MODE = 'none'          # 'none' | 'align' | 'whisper' | 'auto'

# Per note: 'syllable' splits each aligned word at its syllable
# boundaries, so a melisma held over four notes reads la-a-a-hat instead
# of the word `lahat` four times over. 'word' labels every note of a word
# with the whole word.
LABEL_UNIT = 'syllable'      # 'syllable' | 'word'

# Backing vocals in this catalog are sung AT THE SAME TIME as the lead
# (clearest in Mundo, Buwan and Randomantic, where the lyrics sheet puts
# them in parentheses). A forced aligner walks its transcript in order and
# cannot express simultaneity, so handing it both voices forces the
# backing line to be placed after the lead line and every later word
# inherits that error. Keep this False unless you have a reason.
KEEP_PARENTHETICALS = False

# Audio seconds per forward pass through the aligner, plus the context
# carried either side of each chunk and then trimmed off. A whole song at
# once is not an option: self-attention over a 220s track is ~11,000
# frames, which no T4 will hold.
ALIGN_CHUNK_S = None    # seconds of audio per alignment pass
ALIGN_CONTEXT_S = 2.0

# Alignment is bounded to the sung region (voiced_span in Cell 7) plus
# this margin either side. CTC must consume every frame it is handed, and
# the cheapest way to account for a long instrumental outro is to stretch
# the last token over it -- which is exactly what the first aligned run
# did. The margin is there because a word's opening consonant can precede
# the pitch the tracker calls voiced.
ALIGN_MARGIN_S = 0.5

# Note events lying entirely outside that same sung region are dropped.
# Smoothing can bridge sparse instrumental bleed in a vocal stem into
# something that looks like a held note; 7 such notes turned up 5.2s
# before the first sung word.
DROP_NOTES_OUTSIDE_VOICED = True

# A star token matches any audio at any cost, so it can park sound that
# the transcript does not account for. Off by default: it is just as happy
# to swallow a line the singer really did sing, and CTC blanks already
# absorb instrumental intros and outros for free. Turn it on only if the
# alignment report shows drift.
USE_STAR = False

# Whisper, used only for songs with no lyrics on file. 'large-v3' because
# singing is out of distribution for it and model size is the one lever
# that reliably helps; drop to 'medium' on OOM. Nothing downloads it
# unless a song actually needs it.
WHISPER_SIZE = 'large-v3'

# Per song. Fallen is sung in English; the rest of the catalog is Tagalog.
# Telling Whisper the wrong language degrades the whole decode.
LANG_BY_SONG = {'Fallen - Lola Amour': 'en'}
DEFAULT_LANG = 'tl'

# Each song is written as soon as it is extracted, and a song already
# written is skipped. Colab drops sessions, and holding ten songs in
# memory until the end meant a drop at song 7 lost all seven.
RESUME = True

# Put OUT_DIR on Drive to survive a RECYCLED runtime, not just a
# reconnect: /content is wiped when Colab hands you a new VM, uploads
# and all. Costs one consent prompt.
#   from google.colab import drive; drive.mount('/content/drive')
#   OUT_DIR = '/content/drive/MyDrive/huni_ground_truth'
def _vram_gb():
    """VRAM on the allocated GPU, or 0.0 with no GPU."""
    try:
        import torch
        if torch.cuda.is_available():
            return (torch.cuda.get_device_properties(0).total_memory
                    / 1024 ** 3)
    except Exception:                                      # noqa: BLE001
        pass
    return 0.0


def tuned_for_gpu():
    """(torchcrepe batch, alignment chunk seconds) for this GPU.

    Headroom-based, not benchmarked, and deliberately under what the card
    would allow: an OOM in the middle of a ten-song run costs far more
    than a smaller batch. The alignment chunk matters twice over, since
    self-attention over a chunk grows with its square AND every boundary
    is a seam the aligner needs context around.
    """
    gb = _vram_gb()
    if gb >= 32:            # A100 40GB
        return 4096, 60.0
    if gb >= 20:            # L4 24GB
        return 2048, 40.0
    if gb >= 12:            # T4 16GB
        return 512, 20.0
    if gb > 0:              # something smaller
        return 256, 15.0
    return 128, 10.0        # CPU: small, because it is slow either way


_auto_batch, _auto_chunk = tuned_for_gpu()
if BATCH_SIZE is None:
    BATCH_SIZE = _auto_batch
if ALIGN_CHUNK_S is None:
    ALIGN_CHUNK_S = _auto_chunk

OUT_DIR = '/content/ground_truth'


def lang_for(stem):
    return LANG_BY_SONG.get(stem, DEFAULT_LANG)


def prompt_for(stem):
    """Lyrics to bias a Whisper decode with, trimmed to its prompt window.

    Only reached when a song falls back to Whisper. `initial_prompt`
    BIASES the decode toward this vocabulary and spelling without
    constraining it, so a singer who departs from the text is still
    transcribed as what they sang -- but because
    condition_on_previous_text stays False, it conditions the first window
    only. It fixes spelling, not a whole song.
    """
    text = (_sheet_for(stem) or '').strip()
    if not text:
        return None
    text = ' '.join(text.split())
    if len(text) <= 600:
        return text
    cut = text.rfind(' ', 0, 600)      # don't end mid-word
    return text[:cut if cut > 0 else 600]


def _sheet_for(stem):
    """The lyrics sheet for a stem, ignoring capitalisation.

    `IKAW LANG - NOBITA` and `Ikaw Lang - NOBITA` are the same song; a
    separation tool decides which one you get.
    """
    sheets = globals().get('LYRICS_BY_SONG') or {}
    if stem in sheets:
        return sheets[stem]
    low = stem.lower()
    for key, text in sheets.items():
        if key.lower() == low:
            print(f'  note: sheet "{key}" matched stem "{stem}" on case '
                  'alone')
            return text
    return None


def label_mode_for(stem):
    """Which labeller this song gets, and why -> (mode, reason)."""
    has_lyrics = bool((_sheet_for(stem) or '').strip())
    can_align = 'align_words' in globals()
    if LABEL_MODE == 'none':
        return 'none', 'labels off; nothing scores them'
    if LABEL_MODE == 'whisper':
        return 'whisper', 'LABEL_MODE is whisper'
    if LABEL_MODE == 'align':
        if not has_lyrics:
            return 'none', 'no lyrics on file -- run Cell 5'
        if not can_align:
            return 'none', 'aligner not defined -- run Cell 6'
        return 'align', 'LABEL_MODE is align'
    if has_lyrics and can_align:
        return 'align', 'lyrics on file'
    return 'whisper', ('no lyrics on file' if not has_lyrics
                       else 'Cell 6 not run')


print('settings loaded')
_gb = _vram_gb()
print(f'  gpu: {_gb:.0f} GB VRAM -> batch {BATCH_SIZE}, '
      f'{ALIGN_CHUNK_S:.0f}s alignment chunks'
      if _gb else f'  gpu: NONE -- batch {BATCH_SIZE}, '
                  f'{ALIGN_CHUNK_S:.0f}s chunks (CPU will be slow)')
print(f'  pitch: hop {HOP_MS}ms, confidence floor {CONF_MIN}'
      f'{" (auto-relax to " + str(RELAXED_CONF_MIN) + " under " + str(MIN_COVERAGE_PCT) + "% coverage)" if AUTO_RELAX_CONF else ""}, '
      f'{DEFAULT_FMIN_HZ:.0f}-{DEFAULT_FMAX_HZ:.0f} Hz'
      + (f' (overrides: {FMIN_BY_SONG})' if FMIN_BY_SONG else ''))
print(f'  notes: tol {SEMITONE_TOL} semitones, hold {HOLD_MS}ms, '
      f'gap {GAP_MS}ms, min {MIN_NOTE_MS}ms')
if LABEL_MODE == 'none':
    print('  labels: OFF -- notes only, so Cells 5 and 6 can be skipped')
else:
    print(f'  labels: mode={LABEL_MODE}, unit={LABEL_UNIT}, '
          f'parentheticals={"kept" if KEEP_PARENTHETICALS else "dropped"}, '
          f'star={"on" if USE_STAR else "off"}')
print(f'  align: {ALIGN_CHUNK_S:.0f}s chunks, {ALIGN_CONTEXT_S:.0f}s context, {ALIGN_MARGIN_S}s margin')
print(f'  notes outside the sung span: '
      f'{"dropped" if DROP_NOTES_OUTSIDE_VOICED else "kept"}')
print(f'  whisper (fallback only): {WHISPER_SIZE}, '
      f'default lang={DEFAULT_LANG}, overrides={LANG_BY_SONG}')
# LYRICS_BY_SONG lives in Cell 5, which runs after this one, so this
# cell must not depend on it -- re-running settings alone has to work.
_sheets = globals().get('LYRICS_BY_SONG')
print(f'  lyrics on file for '
      f'{sum(1 for v in _sheets.values() if v.strip()) if _sheets else 0} '
      f'song(s)' + ('' if _sheets else ' (Cell 5 not run yet)'))

## Cell 5 — the lyrics *(optional)*

Skip unless `LABEL_MODE` is on. Word labels are never scored, so a default run goes from Cell 4 straight to Cell 7.

Run it when you need word onsets to re-time the app's scrolling lyrics, which is the one thing that genuinely needs them.


In [ ]:
# The lyrics, keyed by the `<Title> - <Artist>` stem that every other
# file for a song already uses.
#
# This is the INPUT to forced alignment, not a guess at it: the words and
# their order come from here, so the only thing Cell 6 estimates is when
# each one is sung. Embedded rather than uploaded so there is a single
# file to carry to Colab and no filename to get wrong. The same text
# lives in backend/lyrics/*.txt, where the repo's validator reads it.
#
# Section markers stay in -- they are stripped on load, and they make the
# repetition structure readable when checking a file against the
# recording by ear. Parenthesised backing vocals stay in too, and
# KEEP_PARENTHETICALS decides what happens to them.
LYRICS_BY_SONG = {
    'Araw-Araw - Ben&Ben': """
Umaga na sa ating duyan
'Wag nang mawawala
Umaga na sa ating duyan
Magmamahal o mahiwaga
Matang magkakilala
Sa unang pagtagpo
Paano dahan-dahang
Sinuyo ang puso
Kay tagal ko nang nag-iisa
Andiyan ka lang pala
Mahiwaga
Pipiliin ka sa araw-araw
Mahiwaga
Ang nadarama sa 'yo'y malinaw
Higit pa sa ligayang
Hatid sa damdamin
Lahat naunawaan
Sa lalim ng tingin
Mahiwaga
Pipiliin ka sa araw-araw
Mahiwaga
Ang nadarama sa 'yo'y malinaw
Sa minsang pagbali ng hangin
Hinila patungo sa akin
Tanging ika'y iibiging wagas at buo
Payapa sa yakap ng iyong hiwaga
Payapa sa yakap ng iyong
Mahiwaga
Pipiliin ka sa araw-araw
Mahiwaga
Ang nadarama sa 'yo'y malinaw
Mahiwaga
'Wag nang mawala araw-araw oh
Mahiwaga
Pipiliin ka araw-araw
""",
    'Buwan - Juan Karlos': """
[Verse 1]
Ako'y sa 'yo, ikaw ay akin
Ganda mo sa paningin
Ako ngayo'y nag-iisa
Sana ay tabihan na

[Chorus]
Sa ilalim ng puting ilaw
Sa dilaw na buwan
Pakinggan mo ang aking sigaw
Sa dilaw na buwan

[Verse 2]
Ayokong mabuhay nang malungkot
Ikaw ang nagpapasaya
At makakasama hanggang sa pagtanda
Halina't tayo'y humiga (Saan kaya?)

[Chorus]
Sa ilalim ng puting ilaw
Sa dilaw na buwan
Pakinggan mo ang aking sigaw
Sa dilaw na buwan, oh

[Bridge]
Ang iyong ganda'y umaabot sa buwan
Ang tibok ng puso'y dinig sa kalawakan
At bumabalik dito sa akin
Ikaw ang mahal, ikaw lang ang mamahalin
Pakinggan ang puso't damdamin
Damdamin aking damdamin

[Chorus]
Sa ilalim ng puting ilaw (Ang iyong ganda'y)
Sa dilaw na buwan (Umaabot sa buwan)
Pakinggan mo ang aking sigaw (Ang tibok ng puso'y)
Sa dilaw na buwan (Rinig sa kalawakan)
Ta-ta-ah
Sa ilalim ng puting ilaw (Ang iyong ganda'y)
Sa dilaw na buwan (Umaabot sa buwan)
La-ah, pakinggan mo ang aking sigaw (Ang tibok ng puso'y)
Sa dilaw na buwan (Rinig sa kalawakan)

[Outro]
La-ah
La-la-la-la-la
La-la-la-la-la-la
Pakinggan, pakingan, pakingan
Pakinggan mo ang aking sigaw o sinta
Sa dilaw na buwan
""",
    'Dadalhin - Regine Velasquez': """
Ang pangarap ko'y
Nagmula sa 'yo
Sa 'yong ganda ang puso'y di makalimot
Tuwing kapiling ka
Tanging nadarama
Ang pagsilip ng bituin sa 'yong mga mata
Ang saya nitong pag-ibig
Sana ay 'di na mag-iiba
Ang pangarap ko ang 'yong binubuhay
Ngayong nagmamahal ka sa akin ng tunay
At ang himig mo'y
Parang musika
Nagpapaligaya sa munting nagwawala
Ang sarap nitong pag-ibig
Lalo na noong sinabi mong
Dadalhin kita sa 'king palasyo
Dadalhin hanggang langit ay manibago
Ang lahat ng ito'y pinangako mo
Dadalhin lang pala ng hangin ang pangarap ko
Nang mawalay ka
Sa aking pagsinta
Bawat saglit gabing lamig ang himig ko
Hanap ay yakap mo
Haplos ng 'yong puso
Parang walang ligtas kundi ang lumuha
Ang hapdi din nitong pag-ibig
Umasa pa sa sinabi mong
Dadalhin kita sa 'king palasyo
Dadalhin hanggang langit ay manibago
Ang lahat ng ito'y pinangako mo
Dadalhin lang pala ng hangin ang pangarap ko
Umiiyak umiiyak ang puso ko
Alaala pa ang sinabi mo
Noong nadarama pa ang pag-ibig mo
Ang lahat ng ito'y pinangako mo ohh
Dadalhin lang pala ng hangin ang pangarap ko oohh
Ang lahat ng ito'y pinangako mo
Dadalhin lang pala ng hangin ang pangarap ko
""",
    'Fallen - Lola Amour': """
What if I told you that I've fallen?
And I like the way you say my name
My heart skips a beat when I hear you calling
And I like that it won't go away
But never mind, don't wanna give you any trouble
Never mind, never mind
I'm okay with being by your side for as long as I can hide
What if I told you that I've fallen?
What if I told you that I've fallen?
A heart shaped arrow through my chest
I'll make you breakfast every morning
And pick you up when you're a mess
I know that it won't ever stop
You know I'll be there when you call me whether you like it or not
Without a warning, now I'm falling for this picture on my phone
But don't mind me I'm just falling, I'll get back up on my own
Please don't say my name, help me put out this flame
I'd rather hold onto this feeling that you don't even believe in
What if I told you that I've fallen?
What if I told you that I've fallen?
(Never mind, never mind, never mind)
What if I told you that I've fallen?
(Never mind, never mind, never mind)
What if I told you that I've fallen?
(Never mind, never mind, never mind)
What if I told you that I've fallen?
(Never mind, never mind, never mind)
What if I told you that I've fallen? (Oh, never mind)
What if I told you that I've fallen? (Oh, never mind)
What if I told you that I've fallen? (Oh, never mind)
What if I told you that I've fallen? (Oh, never mind)
I said never mind (I shouldn't tell you that I've fallen)
""",
    'Ikaw Lang - NOBITA': """
Oh, kay gandang pagmasdan
Ang iyong mga mata
Kumikinang-kinang
'Di ko maintindihan
Ang iyong mga tingin
Labis ang mga ningning
Langit ay bumaba
Bumababa pala ang tala
Tumingin ka sa 'king mga mata
At hindi mo na kailangan pang
Magtanong nang paulit-ulit
Ikaw lang ang iniibig
At kung 'di kumbinsido'y magtiwala ka
Hawakan ang puso't maniwala
Na ikaw lang ang s'yang inibig
Ikaw lang ang iibigin
At sa iyong paglalambing
Ako ay nahulog din
'Di ko alam kung ano ang gagawin
'Di ko alam kung saan titingin
Halik sa labi
Tinginan natin
'Di akalaing
Mahuhulog ka sa 'kin
Tumingin ka sa aking mga mata
At hindi mo na kailangan pang
Magtanong nang paulit-ulit
Ikaw lang ang iniibig
At kung 'di kumbinsido'y magtiwala ka
Hawakan ang puso't maniwala
Na ikaw lang ang s'yang inibig
Ikaw lang ang iibigin, sinta
At sa paglisan ng araw, akala'y 'di ka mahal
At ang nadarama'y hindi magtatagal
Malay ko bang hindi magpapagal
Iibigin kita kahit ga'no pa katagal
Tumingin ka sa 'king mga mata
At hindi mo na kailangan pang
Magtanong nang paulit-ulit
Ikaw lang ang iniibig
At kung 'di kumbinsido'y magtiwala ka
Hawakan ang puso't maniwala
Na ikaw lang ang s'yang inibig
Ikaw lang ang iibigin
La-la-la-la-la-la, la-la-la
La-la-la-la-la-la, la-la-la
La-la-la-la (ah)
La-la-la-la (ah)
La-la-la-la-la-la, la-la-la
Siya lang, la-la, la-la-la
La-la-la-la (ah)
La-la-la-la (ah)
Sinta
""",
    'Kumpas - Moira Dela Torre': """
Pa'no bang mababawi
Lahat ng mga nasabi
Di naman inakalang
Siya'y aalis lang bigla ng walang babala
Sa isang iglap
Nagbago ang lahat
Susuko na dapat nang dumating ka
Ikaw ang kumpas pag naliligaw
Ikaw ang kulay sa langit na bughaw
Sa bawat bagyo na dumadayo
Ikaw ang kanlungan na kailangan ko
Kahit hindi ko alam
Ilang beses mo akong niligtas
Dito sa hantungan ng aming wagas
Pa'nong maniniwala
Wala na sa 'king harapan
Wala naman to sa plano ako'y masaktan ng gan'to
Pa'no na eto
Sa isang iglap
Nalunod ako
Hindi ko na kayang inahon mo ako
Ikaw ang kumpas 'pag naliligaw
Ikaw ang kulay sa langit na bughaw
Sa bawat bagyo na dumadayo
Ikaw ang kanlungan na kailangan ko
At kahit nung di ko alam
Ilang beses ko akong niligtas
Dito sa hantungan ng aming wakas ah
At nung akala ko ubos na
Ikaw ang naging pahinga
Ikaw 'yung kumpas no'ng naliligaw
Ikaw kulay ka sa langit na bughaw
Sa bawat bagyo na dumayo
Ikaw ang kanlungan na nahanap ko
Kahit no'ng 'di ko alam ilang beses mo akong niligtas
Dito sa hantungan at aking wakas
""",
    'Mundo - IV Of Spades': """
San darating ang mga salita
Na nanggagaling sa aming dalawa
Kung lumisan ka wag naman sana
Ika'y kumapit na nang di makawala
Aking sinta ikaw na ang tahanan at mundo
Sa pagbalik mananatili na sa piling mo
Mundo'y magiging ikaw
Wag mag-alala kung nahihirapan ka
Halika na sumama ka
Pagmasdan ang mga tala
Aking sinta ikaw na ang tahanan at mundo
Sa pagbalik mananatili na sa piling mo
Mundo'y magiging ikaw
Limutin na ang mundo
Nang magkasama tayo
Sunod sa bawat galaw
Hindi na maliligaw
Hindi na maliligaw
Hindi na maliligaw
Hindi na maliligaw
Hindi na maliligaw
Hindi na maliligaw
Di na maliligaw
Hindi na maliligaw
Hindi na maliligaw
Hindi na maliligaw
Aking sinta ikaw na ang tahanan at mundo (mundo'y magiging ikaw)
Sa pagbalik mananatili na sa piling mo (mundo'y magiging ikaw)
Aking sinta (limutin na ang mundo)
Ikaw na ang tahanan at mundo (nang magkasama tayo mundo'y magiging ikaw)
Sa pagbalik (sunod sa bawat galaw)
Mananatili na sa piling mo (hindi na maliligaw mundo'y magiging ikaw)
Limutin na ang mundo
Nang magkasama tayo
Sunod sa bawat galaw
Hindi na maliligaw
Mundo'y magiging ikaw
""",
    'Nasa Iyo Na Ang Lahat - Daniel Padilla': """
[Chorus]
Nasa 'yo na ang lahat
Minamahal kita 'pagkat
Nasa 'yo na ang lahat
Pati ang puso ko

Nasa 'yo na ang lahat
Minamahal kitang tapat
Nasa 'yo na ang lahat
Pati ang puso ko

Oh Oh Oh Ohh Ohhh
Na-nasa 'yo na ang lahat

Oh Oh Oh Ohh Ohhh
Na-nasa 'yo na ang lahat

Lahat na mismo nasa 'yo
Ang ganda, ang bait, ang talino
Inggit lahat sila sa 'yo
Kahit pa tapat man kanino

Kaya nung lumapit ka sa 'kin
Ay, bigla akong nahilo
Di akalaing sabihin mong ako na 'yon
Ang hinahanap mo

[Chorus]
Nasa 'yo na ang lahat
Minamahal kita 'pagkat
Nasa 'yo na ang lahat
Pati ang puso ko

Nasa 'yo na ang lahat
Minamahal kitang tapat
Nasa 'yo na ang lahat
Pati ang puso ko

Oh Oh Oh Ohh Ohhh
Na-nasa 'yo na ang lahat

Oh Oh Oh Ohh Ohhh
Na-nasa 'yo na ang lahat

Kinikilig pa rin ako
Ang sarap magmahal 'pag panalo
Nag-iisa sa puso ko
Ito'y kaya 'di na ba magbabago

Ako ang pinili sa dami
Ng ibang nirereto
Hindi akalaing
Sabihin mong ako na lang
Ang kulang sa iyo Ohh

[Chorus]
Nasa 'yo na ang lahat
Minamahal kita 'pagkat
Nasa 'yo na ang lahat
Pati ang puso kohhh

Nasa 'yo na ang lahat
Minamahal kitang tapat
Nasa 'yo na ang lahat
Pati ang puso kohhh

Nasa 'yo na ang lahat
Minamahal kita 'pagkat
Nasa 'yo na ang lahat
Pati ang puso kohhh

Nasa 'yo na ang lahat
Minamahal kitang tapat
Nasa 'yo na ang lahat
Pati ang puso kohhh

Oh Oh Oh Ohh Ohhh
Na-nasa 'yo na ang lahat

Oh Oh Oh Ohh Ohhh
Na-nasa 'yo na ang lahat

[Coda]
Nasa 'yo na ang lahat
""",
    'Pano - Zack Tabudlo': """
Oh giliw naririnig mo ba ng 'yong sarili
Nakakabaliw lumalabas sa 'yong bibig
Alam kong uto uto ako
Alam ko na marupok
Tao lang din naman kasi ako
May nararamdaman din ako
Di kasi manhid na tulad mo
Alam kong sanay bumitaw
Ang isang tulad mo
Lalayo na ba ako
Pano naman ako
Nahulog na sa 'yo
Binitawan mo lang ba talaga ako
Pano naman ako
Naghintay ng matagal sa 'yo
Wala lang ba talaga lahat ng 'yon sa 'yo
Ano na bang gagawin ko
Hmm
Na na na na na oh oh oh
Sinasadya mo ba ang lahat
O trip mo lang ba ako saktan
Pagtapos kong ibigay balikat ko
Pag ika'y umiiyak
Ano bang tingin mo sa akin
Isa ba akong alipin
Wala ka bang modo
Anong ginawa mo
Nagtiwala naman sa 'yo
May nararamdaman din ako
Di kasi manhid na tulad mo
Alam kong sanay bumitaw
Ang isang tulad mo
Lalayo na ba ako
Pano naman ako
Nahulog na sa 'yo
Binitawan mo lang ba talaga ako
Pano naman ako
Naghintay ng matagal sa 'yo
Wala lang ba talaga lahat ng 'yon sa 'yo
Ano na bang gagawin ko
Yah yah yah yah
Oh
Oh
Oh
Uh
Pano naman ako
Nahulog na sa 'yo
Binitawan mo lang ba talaga ako oh
Pano naman ako
Naghintay ng matagal sa 'yo
Wala lang ba talaga lahat ng 'yon sa 'yo
Ano na bang gagawin ko
""",
    'Randomantic - James Reid': """
Kahit wala na
Kahit, kahit wala na
Kahit, kahit wala na
Kahit, kahit wala na
Pa'no kung bigla na lamang
I'll wrap my arms on your hips? Baby, yeah
'Pag 'di mo inaasahan
I'll kiss you right on the lips, baby, yeah
Write little love notes on a post-it
Then I post it on your front door
Show up in suit and tie with a brooch
And with the roses on your front yard, oh-whoa
Doo-doo-doo-doo-doo, doo, yeah
Ang daming paraan, ikaw ang tanging dahilan, oh babe
Haharanahin kita sa'n man tayo magpunta
'Yong tipong kahit wala lang (wala lang)
O kaya biglang may breakfast in bed
Just to keep you interested
Lahat ay dahil wala lang (wala lang)
I just wanna do simple random things to say "I love you"
Kahit wala na
Kahit, kahit wala na (oh, I love you)
Kahit, kahit wala na
Kahit, kahit wala na (ha-ha)
Pilit kang patatawanin (oh)
Kahit na ang sungit mo (kahit na ang sungit mo), oh, yeah
Kahit 'di gano'ng kawangis
Ika'y iguguhit ko, whoa-oh, oh, oh
I'll probably take you to a private little island
Baby, that's right
And maybe share a snack that I like
And, baby, I'll give you the last bite, oh-oh
Doo-doo-doo-doo-doo, doo
Oh, ang daming paraan, ikaw ang tanging dahilan, oh babe
Haharanahin kita sa'n man tayo magpunta
'Yong tipong kahit wala lang (wala lang)
O kaya may on-the-spot dinner date
And this time, I won't be late
Lahat ay dahil wala lang (wala lang)
I just wanna do simple random things to say "I love you"
Randomantic, so romantic
Baby, I'ma let you have it, uh
Doo, ooh
Oh, ang daming paraan, ikaw ang tanging dahilan, oh babe
Haharanahin kita sa'n man tayo magpunta
'Yong tipong kahit wala lang (wala lang)
O kaya 'lilibre ka ng sine
Kahit 'di ko trip ay sige
Lahat ay dahil wala lang (wala lang)
I just wanna do simple random things to say
Haharanahin kita sa'n man tayo magpunta
'Yong tipong kahit wala lang (wala lang)
O kaya iaalay ang kantang 'to
Na sinulat para sa 'yo
Lahat ay dahil wala lang (wala lang)
I just wanna do simple random things to say "I love you"
Kahit wala na
Kahit, kahit wala na (oh-oh, random things to say)
Kahit, kahit wala na ("I love you")
Kahit, kahit wala na (oh baby, baby)
""",
}

print(f'{len(LYRICS_BY_SONG)} lyrics sheet(s) embedded:')
for _stem, _text in sorted(LYRICS_BY_SONG.items()):
    _lines = [l for l in _text.splitlines() if l.strip()
              and not l.strip().startswith('[')]
    print(f'  {_stem[:40]:<40} {len(_lines):3d} lines, '
          f'{sum(len(l.split()) for l in _lines):4d} words')

## Cell 6 — forced alignment *(optional)*

Skip unless `LABEL_MODE` is on; it fetches ~1.2GB of weights the first time Cell 8 aligns anything.

`MMS_FA` is a character-level CTC model covering Tagalog. Given the lyrics it only decides *when* each character is sung, so spelling and word order come from Cell 5 and cannot come out wrong.

It also depends on `to_mono` from Cell 8, which is fine: nothing here is called until Cell 8 runs.


In [ ]:
import re
import unicodedata

import numpy as np
import torch
import torchaudio

# ---- the text layer ----------------------------------------------------
#
# A copy of backend/lyric_text.py, which the repo's validator imports. The
# duplication is deliberate: this notebook has to run on a fresh Colab
# with nothing uploaded but itself, and a scorer that normalised text
# differently from the extractor would report disagreements that exist
# only in the normalisation. backend/test_lyric_text.py covers all of it.

VOWELS = set('aeiou')

# Tagalog's "ng" is one consonant (one letter, pre-1987 alphabet), so it
# can never be split across a syllable boundary.
DIGRAPHS = ('ng',)

_SECTION_RE = re.compile(r'^\s*\[[^\]]*\]\s*$')
_PAREN_RE = re.compile(r'\([^)]*\)')
_TRIM = ',.?!;:"\u201c\u201d\u2026'


def clean_lyrics(text, keep_parentheticals=False):
    """A lyrics sheet -> the lines to align, in order."""
    lines = []
    for raw in text.splitlines():
        if _SECTION_RE.match(raw):
            continue
        line = raw if keep_parentheticals else _PAREN_RE.sub(' ', raw)
        line = ' '.join(line.replace('(', ' ').replace(')', ' ').split())
        if line:
            lines.append(line)
    return lines


def normalize_word(word):
    """A word -> (letters the aligner can consume, index of each in `word`).

    The index list is what makes the round trip possible: character 2 of
    `akoy` is character 3 of `ako'y`, so a span over normalised characters
    can be reported with the apostrophe the singer is actually reading.
    """
    out, idx = [], []
    for i, ch in enumerate(word):
        for part in unicodedata.normalize('NFKD', ch):
            low = part.lower()
            if 'a' <= low <= 'z':
                out.append(low)
                idx.append(i)
    return ''.join(out), idx


def syllabify(norm):
    """Normalised letters -> one (start, end) character span per syllable.

        V|V     two adjacent vowels split            i|yo
        V|CV    one consonant onsets the next        na|sa, la|hat
        VC|CV   two or more, the first is a coda     pag|kat
    """
    if not norm:
        return []
    units, positions, i = [], [], 0
    while i < len(norm):
        for d in DIGRAPHS:
            if norm.startswith(d, i):
                units.append(d)
                positions.append(i)
                i += len(d)
                break
        else:
            units.append(norm[i])
            positions.append(i)
            i += 1
    vowel_at = [u in VOWELS for u in units]
    if not any(vowel_at):
        return [(0, len(norm))]
    starts = [0]
    vowels = [k for k, v in enumerate(vowel_at) if v]
    for a, b in zip(vowels, vowels[1:]):
        gap = b - a - 1
        cut = b if gap == 0 else (b - 1 if gap == 1 else a + 2)
        if cut > starts[-1]:
            starts.append(cut)
    spans = []
    for k, s in enumerate(starts):
        e = starts[k + 1] if k + 1 < len(starts) else len(units)
        spans.append((positions[s],
                      positions[e] if e < len(units) else len(norm)))
    return spans


class LyricWord:
    """One word, in both spellings, with its syllables."""

    __slots__ = ('text', 'norm', 'syllables', 'line')

    def __init__(self, text, line=0):
        self.text = text
        self.line = line
        self.norm, idx = normalize_word(text)
        # A hyphen marks a morpheme boundary that is also a syllable
        # boundary: nag-iisa is nag-i-i-sa, not the na-gi-i-sa the
        # open-syllable rules alone would give.
        spans, offset = [], 0
        for part in re.split(r'[-\u2010-\u2015]', text):
            pnorm, _ = normalize_word(part)
            if not pnorm:
                continue
            spans += [(s + offset, e + offset) for s, e in syllabify(pnorm)]
            offset += len(pnorm)
        self.syllables = []
        for k, (s, e) in enumerate(spans):
            lo = 0 if k == 0 else idx[s]
            self.syllables.append((text[lo:idx[e - 1] + 1], s, e))

    @property
    def alignable(self):
        return bool(self.norm)


def lyric_words(text, keep_parentheticals=False):
    """A lyrics sheet -> the alignable words, in order."""
    words = []
    for n, line in enumerate(clean_lyrics(text, keep_parentheticals)):
        for token in line.split():
            w = LyricWord(token.strip(_TRIM), n)
            if w.alignable:
                words.append(w)
    return words


# to_mono lives in Cell 8, which always runs -- this one does not.

# ---- the forced aligner ------------------------------------------------

ALIGN_BUNDLE = torchaudio.pipelines.MMS_FA
_ALIGN = {}


def load_aligner(device):
    """MMS_FA, loaded once. ~1.2GB of weights on first call."""
    if not _ALIGN:
        print(f'  loading the MMS forced aligner on {device} '
              '(first call downloads ~1.2GB)...')
        _ALIGN['model'] = ALIGN_BUNDLE.get_model(
            with_star=USE_STAR).to(device).eval()
        _ALIGN['tokenizer'] = ALIGN_BUNDLE.get_tokenizer()
        _ALIGN['aligner'] = ALIGN_BUNDLE.get_aligner()
    return _ALIGN['model'], _ALIGN['tokenizer'], _ALIGN['aligner']


@torch.inference_mode()
def emission_for(wav, model, device, t_offset=0.0):
    """Log-probs over characters -> (frames x vocab, time of each frame).

    Chunked, because a whole song in one pass is not an option: a 220s
    track is ~11,000 frames and self-attention over that will not fit a
    T4. Each chunk is run with ALIGN_CONTEXT_S of real audio either side
    and those frames are then trimmed off, so a chunk boundary does not
    land on a frame that had no left or right context.

    Frame times are built per chunk from that chunk's own frame count
    rather than from an assumed stride, so the mapping stays correct
    whatever the convolution stack rounds to. `t_offset` is where `wav`
    starts in the original track, so the times come back on the track's
    own clock even when only part of it was handed over.
    """
    sr = ALIGN_BUNDLE.sample_rate
    n = wav.size(1)
    chunk = max(sr, int(ALIGN_CHUNK_S * sr))
    ctx = int(ALIGN_CONTEXT_S * sr)

    pieces, times = [], []
    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        lo, hi = max(0, start - ctx), min(n, end + ctx)
        em, _ = model(wav[:, lo:hi].to(device))
        em = em[0]
        t = em.size(0)
        a = round(t * (start - lo) / (hi - lo))
        b = max(round(t * (end - lo) / (hi - lo)), a + 1)
        keep = em[a:b].float().cpu()
        pieces.append(keep)
        step = (end - start) / keep.size(0) / sr
        times.append(t_offset + start / sr
                     + np.arange(keep.size(0)) * step)
    return torch.cat(pieces, dim=0), np.concatenate(times)


def align_words(path, lyrics, device, t_lo=None, t_hi=None):
    """Forced-align a lyrics sheet to a vocal stem.

    Returns (word_spans, syllable_spans, report). Both span lists are in
    the shape label_notes() consumes -- {'text', 'start', 'end'} -- so
    LABEL_UNIT only chooses which list to hand it.

    `t_lo`/`t_hi` bound the audio to the part that is actually sung, taken
    from the pitch track by voiced_span(). This is not an optimisation.
    CTC has to consume every frame it is given, and the cheapest way to
    account for a long instrumental outro is to stretch the final token
    across it -- which is why the first aligned run put the last word of
    Nasa Iyo Na Ang Lahat at 220.8s on a track that stops singing at
    209.2s. Handing over only the sung region removes the silence there
    was to absorb, and gives the alignment less room to drift.
    """
    words = lyric_words(lyrics, KEEP_PARENTHETICALS)
    if not words:
        raise ValueError('no alignable words in the lyrics')

    wav, sr = torchaudio.load(path)
    wav = to_mono(wav)
    if sr != ALIGN_BUNDLE.sample_rate:
        wav = torchaudio.functional.resample(wav, sr,
                                             ALIGN_BUNDLE.sample_rate)
    sr = ALIGN_BUNDLE.sample_rate

    # A little margin either side: a word's first consonant can start
    # before the pitch tracker calls the note voiced.
    total = wav.size(1) / sr
    lo = 0.0 if t_lo is None else max(0.0, t_lo - ALIGN_MARGIN_S)
    hi = total if t_hi is None else min(total, t_hi + ALIGN_MARGIN_S)
    if hi - lo < 1.0:                    # nothing usable; align it all
        lo, hi = 0.0, total
    wav = wav[:, int(lo * sr):int(hi * sr)]

    model, tokenizer, aligner = load_aligner(device)
    emission, frame_times = emission_for(wav, model, device, t_offset=lo)

    tokens = tokenizer([w.norm for w in words])
    token_spans = aligner(emission, tokens)

    def at(frame):
        """Frame index -> seconds, clamped to the track."""
        return float(frame_times[min(max(frame, 0), len(frame_times) - 1)])

    word_spans, syll_spans = [], []
    scores, split_failed = [], 0
    for w, spans in zip(words, token_spans):
        if not spans:
            continue
        start, end = at(spans[0].start), at(spans[-1].end)
        if end <= start:
            end = start + 0.01
        score = float(np.mean([s.score for s in spans]))
        scores.append(score)
        word_spans.append({'text': w.text, 'start': start, 'end': end,
                           'score': score})

        # One token per normalised character is what lets a word be cut
        # into syllables. If the tokenizer dropped a character, that
        # assumption is broken for this word, so fall back to labelling
        # the whole word rather than cutting in the wrong place.
        if len(spans) != len(w.norm):
            split_failed += 1
            syll_spans.append({'text': w.text, 'start': start, 'end': end,
                               'score': score})
            continue
        for text, s, e in w.syllables:
            sy = spans[s:e]
            if not sy:
                continue
            ss, se = at(sy[0].start), at(sy[-1].end)
            syll_spans.append({'text': text, 'start': ss,
                               'end': se if se > ss else ss + 0.01,
                               'score': float(np.mean(
                                   [x.score for x in sy]))})

    report = {
        'audio_from': lo,
        'audio_to': hi,
        'words_in_lyrics': len(words),
        'words_aligned': len(word_spans),
        'syllables': len(syll_spans),
        'mean_score': float(np.mean(scores)) if scores else 0.0,
        'weak': sum(1 for s in scores if s < 0.30),
        'split_failed': split_failed,
        'first': word_spans[0]['start'] if word_spans else None,
        'last': word_spans[-1]['end'] if word_spans else None,
        'frames': int(emission.size(0)),
    }
    return word_spans, syll_spans, report


print('forced aligner loaded (weights fetch on first use)')
print(f'  {len(LYRICS_BY_SONG)} sheet(s) available; '
      f'parentheticals {"kept" if KEEP_PARENTHETICALS else "dropped"}')
for _stem in sorted(LYRICS_BY_SONG):
    _w = lyric_words(LYRICS_BY_SONG[_stem], KEEP_PARENTHETICALS)
    print(f'  {_stem[:40]:<40} {len(_w):4d} words -> '
          f'{sum(len(x.syllables) for x in _w):4d} syllables')

## Cell 7 — note segmentation

The part that replaces dividing a word span by its vowel count. Tested
against synthetic contours with known answers before being put here: a
held note with vibrato stays one note, three real notes inside one
syllable come out as three, a 300 ms rest splits a note while a 30 ms
dropout does not, seven equal slices of one pitch collapse to one event,
and a 600 ms slide yields a few notes rather than dozens.

In [ ]:
import numpy as np

NOTE_NAMES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#',
              'B']


def hz_to_midi(hz):
    """Hz -> float MIDI. Non-positive input becomes NaN, not an error."""
    hz = np.asarray(hz, dtype=float)
    out = np.full(hz.shape, np.nan)
    good = hz > 0
    out[good] = 69.0 + 12.0 * np.log2(hz[good] / 440.0)
    return out


def midi_to_note(midi):
    m = int(round(midi))
    return f'{NOTE_NAMES[m % 12]}{m // 12 - 1}'


def running_median(x, window, min_valid=None):
    """NaN-aware median filter, same length out as in.

    Median rather than mean: it removes tracker jitter and isolated octave
    jumps without smearing a real onset across the boundary.

    `min_valid` is the fix for a measured defect. Taking the median of
    whatever happens to be valid in the window means ONE confident frame
    in five makes all five voiced, so the filter was inventing voicing out
    of sparse, weakly-periodic instrumental bleed in the vocal stem --
    which is how 7 notes appeared 5.2s before Nasa Iyo Na Ang Lahat's
    first sung word. Requiring most of the window to be real pitch before
    believing any of it leaves genuine singing untouched (a sung note
    fills its window) and leaves bleed as the silence it is.
    """
    if window <= 1:
        return np.asarray(x, dtype=float)
    half = window // 2
    need = (window + 1) // 2 if min_valid is None else min_valid
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    for i in range(len(x)):
        lo, hi = max(0, i - half), min(len(x), i + half + 1)
        chunk = x[lo:hi]
        chunk = chunk[~np.isnan(chunk)]
        # Scale the requirement at the edges, where the window is clipped
        # and can never be full.
        edge_need = max(1, int(round(need * (hi - lo) / window)))
        out[i] = np.median(chunk) if len(chunk) >= edge_need else np.nan
    return out


def voiced_span(times, hz, periodicity, sustain_ms=150):
    """(first, last) time a sustained run of confident pitch covers.

    Deliberately stricter than "any confident frame": a single frame is
    noise, and this is used to decide what counts as part of the
    performance at all. A run has to last `sustain_ms` to qualify, which
    is already shorter than any note worth scoring.

    Returns (None, None) when nothing qualifies, so callers can tell
    "no singing found" from "singing starts at 0.0".
    """
    times = np.asarray(times, dtype=float)
    ok = (np.asarray(periodicity, dtype=float) >= CONF_MIN) & \
         (np.asarray(hz, dtype=float) > 0)
    if not ok.any():
        return None, None
    hop_ms = (times[1] - times[0]) * 1000 if len(times) > 1 else HOP_MS
    need = max(1, int(round(sustain_ms / hop_ms)))

    # Longest-run bookkeeping in one pass: the first and last frame that
    # belong to any run of at least `need` consecutive confident frames.
    first = last = None
    run = 0
    for i, v in enumerate(ok):
        if v:
            run += 1
            if run >= need:
                if first is None:
                    first = i - run + 1
                last = i
        else:
            run = 0
    if first is None:
        return None, None
    return float(times[first]), float(times[last] + hop_ms / 1000.0)


def notes_within(notes, t_lo, t_hi, pad=0.30):
    """Drops note events that lie outside the sung span -> (kept, dropped).

    A note entirely before the first sustained pitch or after the last one
    is not part of the performance. Returned rather than silently removed
    so the count shows up in the extract log: a large number means the
    vocal stem has bleed worth looking at, not that this filter is doing
    its job.
    """
    if t_lo is None or t_hi is None:
        return list(notes), []
    kept, dropped = [], []
    for n in notes:
        (kept if n['end'] > t_lo - pad and n['start'] < t_hi + pad
         else dropped).append(n)
    return kept, dropped


def segment_notes(times, hz, periodicity):
    """Splits an f0 contour into note events: [{'start','end','midi',...}].

    A new note begins when the smoothed pitch leaves the current note's
    running median by more than SEMITONE_TOL and STAYS there for HOLD_MS.
    Requiring persistence is what keeps a slide or a moment of tracker
    noise from becoming a note. A voicing gap over GAP_MS also ends a
    note, because a breath or rest is a real boundary.

    Notes outside MIDI_LO..MIDI_HI are DROPPED, never "corrected":
    inventing a pitch would be worse than admitting a frame was unusable.
    """
    times = np.asarray(times, dtype=float)
    hop_ms = (times[1] - times[0]) * 1000 if len(times) > 1 else HOP_MS
    hold_frames = max(1, int(round(HOLD_MS / hop_ms)))
    gap_frames = max(1, int(round(GAP_MS / hop_ms)))
    smooth_window = max(1, int(round(SMOOTH_MS / hop_ms)))

    midi = hz_to_midi(hz)
    midi[np.asarray(periodicity, dtype=float) < CONF_MIN] = np.nan
    smooth = running_median(midi, smooth_window)

    notes = []
    cur, pending = [], []
    unvoiced_run = 0

    def close(frames):
        if not frames:
            return
        vals = smooth[frames]
        vals = vals[~np.isnan(vals)]
        if len(vals) == 0:
            return
        notes.append({
            'start': float(times[frames[0]]),
            'end': float(times[frames[-1]] + hop_ms / 1000.0),
            'midi': float(np.median(vals)),
            'frames': len(frames),
        })

    for i in range(len(smooth)):
        if np.isnan(smooth[i]):
            unvoiced_run += 1
            if unvoiced_run >= gap_frames:
                close(cur + pending)
                cur, pending = [], []
            continue
        unvoiced_run = 0

        if not cur:
            cur, pending = [i], []
            continue

        ref = np.nanmedian(smooth[cur])
        if abs(smooth[i] - ref) <= SEMITONE_TOL:
            cur.extend(pending)       # the deviation was a wobble
            pending = []
            cur.append(i)
            continue

        pending.append(i)
        if len(pending) < hold_frames:
            continue

        # It held, so the note really changed. The boundary belongs where
        # the deviation STARTED, not where it was confirmed.
        close(cur)
        cur, pending = pending, []

    close(cur + pending)

    kept = [n for n in notes
            if (n['end'] - n['start']) * 1000 >= MIN_NOTE_MS
            and MIDI_LO <= n['midi'] <= MIDI_HI]

    # Join neighbours left at the same pitch with no gap: one held note
    # that a brief wobble split in two.
    merged = []
    for n in kept:
        if (merged and abs(n['midi'] - merged[-1]['midi']) <= SEMITONE_TOL
                and n['start'] - merged[-1]['end'] <= GAP_MS / 1000.0):
            prev = merged[-1]
            total = prev['frames'] + n['frames']
            prev['midi'] = (prev['midi'] * prev['frames']
                            + n['midi'] * n['frames']) / total
            prev['end'] = n['end']
            prev['frames'] = total
        else:
            merged.append(dict(n))
    return merged


def label_notes(notes, words):
    """Attaches the overlapping word to each note. A LABEL, not a target.

    Words are never scored against what the singer sang; the text is here
    so a developer can see at a glance whether a ground-truth file lines
    up with the song. A note with no overlapping word is left blank rather
    than guessed.
    """
    out = []
    for n in notes:
        mid = 0.5 * (n['start'] + n['end'])
        best, best_overlap = '', 0.0
        for w in words:
            overlap = min(n['end'], w['end']) - max(n['start'], w['start'])
            if overlap > best_overlap:
                best, best_overlap = w['text'], overlap
        if not best and words:
            nearest = min(words, key=lambda w: abs(
                0.5 * (w['start'] + w['end']) - mid))
            if abs(0.5 * (nearest['start'] + nearest['end']) - mid) < 0.5:
                best = nearest['text']
        out.append({'syllable': best,
                    'start': round(n['start'], 3),
                    'end': round(n['end'], 3),
                    'note': midi_to_note(n['midi'])})
    return out


print('note segmentation loaded')
print(f'  smoothing needs {(max(1, int(round(SMOOTH_MS / HOP_MS))) + 1) // 2} of {max(1, int(round(SMOOTH_MS / HOP_MS)))} frames to be real pitch')

## Cell 8 — extract

torchcrepe for pitch, Whisper for word timings, both over the isolated
vocals. No alignment step — the timestamps are already on the
instrumental's timeline. Each song is caught separately so one failure
does not lose the rest.

In [ ]:
import gc
import json
import os
import traceback

import numpy as np
import torch
import torchcrepe

os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

RESULTS = {}
VOCAL_ONSET = {}
VOCAL_OFFSET = {}
ALIGN_REPORT = {}
WORD_SPANS = {}
LABEL_SOURCE = {}

_WHISPER = {}


def whisper_model():
    """Loaded on first use, so an all-aligned run never downloads 3GB."""
    if not _WHISPER:
        import whisper
        print(f'  loading whisper-{WHISPER_SIZE} on {DEVICE} '
              '(only needed for songs with no lyrics)...')
        _WHISPER['m'] = whisper.load_model(WHISPER_SIZE, device=DEVICE)
    return _WHISPER['m']


def to_mono(audio):
    """Shapes loaded audio into the (1, samples) the models need.

    A stereo stem loads as (2, samples) and torchcrepe's internal
    resampler calls .squeeze(0) on it, failing with "cannot select an axis
    to squeeze out which has size not equal to one". Separated stems are
    usually stereo, so this is the normal case.

    Channels are averaged for the better signal-to-noise, but averaging
    cancels anything phase-inverted between them, which would hand the
    model silence. So the mix is compared against the individual channels
    and the loudest channel used instead if averaging lost most of it.
    """
    if audio.dim() == 1:
        audio = audio.unsqueeze(0)
    if audio.shape[0] == 1:
        return audio
    mixed = audio.mean(dim=0, keepdim=True)
    rms_mixed = float(mixed.pow(2).mean().sqrt())
    per_channel = [float(audio[c].pow(2).mean().sqrt())
                   for c in range(audio.shape[0])]
    loudest = max(range(len(per_channel)), key=lambda c: per_channel[c])
    if rms_mixed < 0.5 * per_channel[loudest]:
        print(f'    channels cancel when averaged (mix RMS {rms_mixed:.5f} '
              f'vs channel {loudest} {per_channel[loudest]:.5f}); '
              f'using channel {loudest}')
        return audio[loudest:loudest + 1]
    return mixed

def pitch_track(path, fmin=None, fmax=None):
    """torchcrepe over the vocal stem -> (times, hz, periodicity)."""
    audio, sr = torchcrepe.load.audio(path)
    print(f'    loaded {tuple(audio.shape)} at {sr} Hz')
    audio = to_mono(audio)
    # hop_length is given at the FILE's sample rate; torchcrepe converts
    # it to its own 16 kHz internally, so HOP_MS stays the real hop.
    hop = int(sr * HOP_MS / 1000)
    hz, periodicity = torchcrepe.predict(
        audio, sr, hop_length=hop,
        fmin=fmin if fmin is not None else DEFAULT_FMIN_HZ,
        fmax=fmax if fmax is not None else DEFAULT_FMAX_HZ,
        model='full', batch_size=BATCH_SIZE, device=DEVICE,
        return_periodicity=True)
    hz = hz.squeeze().cpu().numpy()
    periodicity = periodicity.squeeze().cpu().numpy()
    times = np.arange(len(hz)) * HOP_MS / 1000.0
    return times, hz, periodicity


def word_spans(path, lang, prompt=None):
    """Whisper word timings over the isolated vocals. Fallback only.

    Reached only for a song with no lyrics on file. Words LABEL notes and
    are never scored, so a wrong word costs a label and not a target --
    but on this catalog it costs a lot of them: 27% WER, measured.

    condition_on_previous_text stays False: singing is slow and repetitive,
    which is exactly the condition where carrying one segment's output into
    the next turns into invented repetition. The cost is that
    `initial_prompt` then conditions only the first window, so a lyrics
    prompt mostly fixes spelling and vocabulary rather than the whole song.
    """
    tr = whisper_model().transcribe(path, language=lang,
                                    word_timestamps=True,
                                    condition_on_previous_text=False,
                                    initial_prompt=prompt)
    spans = []
    for seg in tr.get('segments', []):
        for w in seg.get('words', []):
            text = w.get('word', '').strip()
            if not text:
                continue
            s, e = float(w['start']), float(w['end'])
            if e > s:
                spans.append({'text': text, 'start': s, 'end': e})
    return spans


def _written_path(stem):
    return os.path.join(OUT_DIR, f'{stem}.json')


def _load_written(stem):
    """A song written by an earlier run, so it is not redone."""
    try:
        with open(_written_path(stem), encoding='utf-8') as fh:
            return json.load(fh)['data']
    except Exception:            # noqa: BLE001
        return None


for stem, (vocals_path, instr_path) in PAIRS.items():
    print(f'\n{"=" * 70}\n{stem}\n{"=" * 70}')
    if RESUME and stem not in RESULTS:
        _done = _load_written(stem)
        if _done:
            RESULTS[stem] = _done
            LABEL_SOURCE.setdefault(stem, 'none')
            print(f'  already extracted ({len(_done)} notes) -- skipping. '
                  'Set RESUME = False to redo it.')
            continue
    if stem in RESULTS:
        print('  already in RESULTS this session -- skipping.')
        continue
    try:
        fmin, fmax = fmin_for(stem), fmax_for(stem)
        print(f'  pitch tracking the vocals '
              f'({fmin:.0f}-{fmax:.0f} Hz)...')
        times, hz, periodicity = pitch_track(vocals_path, fmin, fmax)
        onset, offset = voiced_span(times, hz, periodicity)
        VOCAL_ONSET[stem] = onset
        VOCAL_OFFSET[stem] = offset
        if onset is None:
            print('    no sustained pitch found; skipping')
            continue
        print(f'    singing runs {onset:.2f}s .. {offset:.2f}s')

        # segment_notes and voiced_span read CONF_MIN as a global, so the
        # per-song value is bound here rather than threaded through every
        # signature. Restored after the song so one override cannot leak
        # into the next.
        _conf_was = CONF_MIN
        CONF_MIN = conf_min_for(stem)
        print(f'  segmenting notes (confidence floor {CONF_MIN})...')
        note_events = segment_notes(times, hz, periodicity)

        def _coverage(evs):
            if not evs:
                return 0.0
            span = max(e['end'] for e in evs) - min(e['start'] for e in evs)
            voiced = sum(e['end'] - e['start'] for e in evs)
            return 100.0 * voiced / span if span > 0 else 0.0

        cov = _coverage(note_events)
        if AUTO_RELAX_CONF and cov < MIN_COVERAGE_PCT \
                and CONF_MIN > RELAXED_CONF_MIN:
            print(f'    only {cov:.0f}% of the span is inside a note; '
                  f'retrying at {RELAXED_CONF_MIN}')
            CONF_MIN = RELAXED_CONF_MIN
            retry = segment_notes(times, hz, periodicity)
            cov2 = _coverage(retry)
            print(f'    {len(note_events)} notes / {cov:.0f}% -> '
                  f'{len(retry)} notes / {cov2:.0f}%')
            if cov2 > cov:
                note_events, cov = retry, cov2
                print(f'    keeping the relaxed pass (check Cell 9 for the '
                      'octave-error count that comes with it)')
            else:
                CONF_MIN = _conf_was
                print('    no better; keeping the original')
        print(f'    {cov:.0f}% of the span is inside a note')
        if DROP_NOTES_OUTSIDE_VOICED:
            note_events, outside = notes_within(note_events, onset, offset)
            if outside:
                print(f'    dropped {len(outside)} note(s) outside the sung '
                      f'span ({outside[0]["start"]:.1f}s .. '
                      f'{outside[-1]["end"]:.1f}s) -- bleed in the stem, '
                      'not singing')
        print(f'    {len(note_events)} note events')
        if not note_events:
            print('    nothing segmented; skipping')
            continue

        mode, why = label_mode_for(stem)
        LABEL_SOURCE[stem] = mode
        print(f'  labels: {mode} ({why})')

        if mode == 'align':
            word_list, syll_list, rep = align_words(
                vocals_path, _sheet_for(stem), DEVICE,
                t_lo=onset, t_hi=offset)
            ALIGN_REPORT[stem] = rep
            print(f'    {rep["words_aligned"]}/{rep["words_in_lyrics"]} '
                  f'words placed over {rep["frames"]} frames, '
                  f'{rep["syllables"]} syllables')
            print(f'    mean confidence {rep["mean_score"]:.2f}, '
                  f'{rep["weak"]} word(s) below 0.30'
                  + (f', {rep["split_failed"]} not syllabified'
                     if rep['split_failed'] else ''))
            print(f'    aligned inside {rep["audio_from"]:.2f}s .. '
                  f'{rep["audio_to"]:.2f}s; lyrics landed '
                  f'{rep["first"]:.2f}s .. {rep["last"]:.2f}s')
            labels = syll_list if LABEL_UNIT == 'syllable' else word_list
            # Kept whatever LABEL_UNIT is: word onsets are what time the
            # app's scrolling lyrics, and nothing else measures when a
            # word was actually sung.
            WORD_SPANS[stem] = word_list
        elif mode == 'whisper':
            lang = lang_for(stem)
            prompt = prompt_for(stem)
            print(f'    free transcription (lang={lang}'
                  f'{", prompted" if prompt else ""}) -- expect ~27% WER')
            labels = word_spans(vocals_path, lang, prompt)
            print(f'    {len(labels)} words')
            # Printed so the transcription can be judged directly, rather
            # than only through the labels it ends up producing.
            heard = ' '.join(w['text'] for w in labels[:25])
            print(f'    heard: {heard[:160]}')
        else:
            labels = []
            print('    no labels; notes are still extracted and scored')

        rows = label_notes(note_events, labels)
        unlabelled = sum(1 for r in rows if not r['syllable'])
        print(f'    {len(rows)} rows, {unlabelled} without a label')
        print(f'    spans {rows[0]["start"]:.2f}s .. '
              f'{max(r["end"] for r in rows):.2f}s')
        RESULTS[stem] = rows

        # Written now, not in Cell 10: a disconnect after this point costs
        # nothing, and Cell 8 can be re-run to carry on from here.
        os.makedirs(OUT_DIR, exist_ok=True)
        with open(_written_path(stem), 'w', encoding='utf-8') as fh:
            json.dump({'data': rows}, fh, ensure_ascii=False, indent=1)
        print(f'    saved {stem}.json')
        if WORD_SPANS.get(stem):
            with open(os.path.join(OUT_DIR, f'{stem}.words.json'), 'w',
                      encoding='utf-8') as fh:
                json.dump({'stem': stem,
                           'source': 'MMS_FA forced alignment of the '
                                     'vocal stem',
                           'words': [{'text': w['text'],
                                      'start': round(w['start'], 3),
                                      'end': round(w['end'], 3),
                                      'score': round(w.get('score', 0.0), 4)}
                                     for w in WORD_SPANS[stem]]},
                          fh, ensure_ascii=False, indent=1)
            print(f'    saved {stem}.words.json')

    except Exception:
        print(f'  FAILED on {stem}:')
        traceback.print_exc()
    finally:
        try:
            CONF_MIN = _conf_was
        except NameError:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

_pending = [s for s in PAIRS if s not in RESULTS]
print(f'\n{len(RESULTS)} of {len(PAIRS)} song(s) done, written to '
      f'{OUT_DIR}.')
if _pending:
    print(f'  {len(_pending)} still to do: {", ".join(_pending[:4])}'
          + (' ...' if len(_pending) > 4 else ''))
    print('  Re-run this cell to continue; finished songs are skipped.')
print(f'  {sum(1 for m in LABEL_SOURCE.values() if m == "align")} '
      'forced-aligned from lyrics, '
      f'{sum(1 for m in LABEL_SOURCE.values() if m == "whisper")} '
      'transcribed by Whisper.')

## Cell 9 — validate

Every check here corresponds to a defect that was actually measured in a
previous run, including the two this rewrite exists to fix: equal-duration
splits and same-pitch runs. **Do not ship a song that fails one.**

In [ ]:
import re

from mutagen import File as MutagenFile

NOTE_BASE = {'C': 0, 'D': 2, 'E': 4, 'F': 5, 'G': 7, 'A': 9, 'B': 11}


def note_to_midi(name):
    m = re.fullmatch(r'([A-G])([#b]?)(-?\d+)', (name or '').strip())
    if not m:
        return None
    semi = NOTE_BASE[m.group(1)]
    semi += 1 if m.group(2) == '#' else (-1 if m.group(2) == 'b' else 0)
    return 12 * (int(m.group(3)) + 1) + semi


def note_name(midi):
    """MIDI -> 'C4', for reporting the register a song actually used."""
    names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    m = int(round(midi))
    return f'{names[m % 12]}{m // 12 - 1}'


print(f"{'song':<30} {'rows':>5} {'ends':>7} {'audio':>7} {'1st':>5} "
      f"{'onset':>6} {'rng':>4} {'runs':>5} {'equal':>6} {'med s':>6}")
print('-' * 104)

OK_SONGS = []
for stem, rows in sorted(RESULTS.items()):
    _, instr_path = PAIRS[stem]
    audio_dur = float(MutagenFile(instr_path).info.length)
    ends = max(r['end'] for r in rows)
    first = rows[0]['start']
    onset = VOCAL_ONSET.get(stem)

    midis = [m for m in (note_to_midi(r['note']) for r in rows) if m]
    spread = (max(midis) - min(midis)) if midis else 0
    outliers = sum(1 for m in midis if m < 40 or m > 84)

    srt = sorted(rows, key=lambda r: r['start'])
    overlap = sum(1 for a, b in zip(srt, srt[1:])
                  if b['start'] < a['end'] - 1e-9)

    # The two defects this rewrite targets.
    same_pitch_runs = sum(1 for a, b in zip(srt, srt[1:])
                          if a['note'] == b['note']
                          and b['start'] - a['end'] < 0.12)
    durs = sorted(r['end'] - r['start'] for r in rows)
    med = durs[len(durs) // 2]
    rounded = [round(d, 3) for d in durs]
    # Arithmetic splits show up as the same duration repeating a lot.
    most_common = max(rounded.count(d) for d in set(rounded))
    equal_share = 100 * most_common / len(rounded)

    problems = []
    # Printed after this song's row, so the table does not read off by one.
    notices = []
    if ends > audio_dur + 0.5:
        problems.append(f'runs {ends - audio_dur:.1f}s past the audio')
    if onset is not None and first < onset - 2.0:
        problems.append(f'first note {onset - first:.1f}s before any singing '
                        '-- set DROP_NOTES_OUTSIDE_VOICED and re-run Cell 8')
    offset = VOCAL_OFFSET.get(stem)
    if offset is not None and ends > offset + 2.0:
        problems.append(f'last note {ends - offset:.1f}s after the singing '
                        'stops')
    if spread > 40:
        problems.append(f'{spread} semitone range (octave errors?)')
    if outliers:
        problems.append(f'{outliers} notes outside a singable range')
    if overlap:
        problems.append(f'{overlap} overlapping notes')
    if same_pitch_runs > len(rows) * 0.10:
        problems.append(f'{same_pitch_runs} adjacent same-pitch notes '
                        '(still over-segmented)')
    if equal_share > 20:
        problems.append(f'{equal_share:.0f}% of notes share one duration '
                        '(arithmetic splits?)')
    if len(rows) < 50:
        problems.append(f'only {len(rows)} notes')

    # Octave errors, against each note's own neighbours. A note half the
    # true frequency lands 12 semitones low and still sits inside a
    # singable range, so the spread/outlier checks above cannot see it.
    octave_off = []
    for k, r in enumerate(srt):
        m = note_to_midi(r['note'])
        if m is None:
            continue
        near = [note_to_midi(o['note']) for j, o in enumerate(srt)
                if j != k and abs(o['start'] - r['start']) <= 4.0]
        near = [x for x in near if x is not None]
        if len(near) < 2:
            continue
        local = sorted(near)[len(near) // 2]
        if 10.5 <= abs(m - local) <= 13.5:
            octave_off.append(r)
    if octave_off:
        share = 100 * len(octave_off) / len(rows)
        by_syl = {}
        for r in octave_off:
            key = (r.get('syllable') or '').strip()
            if key:
                by_syl[key] = by_syl.get(key, 0) + 1
        worst = sorted(by_syl.items(), key=lambda kv: -kv[1])[:3]
        detail = ('; worst: '
                  + ', '.join(f'{k!r}x{v}' for k, v in worst if v > 1)
                  if any(v > 1 for _, v in worst) else '')
        msg = (f'{len(octave_off)} note(s) ~12 semitones off their own '
               f'neighbours ({share:.0f}%) -- likely octave errors'
               + detail)
        if share > 3:
            problems.append(msg + f'; raise fmin above {fmin_for(stem):.0f} Hz')
        else:
            notices.append(msg)

    # The register actually used, so the next run's fmin can be chosen
    # from evidence instead of guessed.
    if midis:
        voiced_lo = min(midis)
        notices.append(
            f'register {note_name(voiced_lo)}-{note_name(max(midis))}, '
            f'{440.0 * 2 ** ((voiced_lo - 69) / 12):.0f}'
            f'-{440.0 * 2 ** ((max(midis) - 69) / 12):.0f} Hz '
            f'(fmin {fmin_for(stem):.0f}, fmax {FMAX_HZ:.0f})')

    # A song sings through most of the stretch it spans. Far less than
    # that means the tracker found almost nothing, which no other column
    # here would reveal.
    voiced_s = sum(r['end'] - r['start'] for r in rows)
    span_s = ends - first
    coverage = 100.0 * voiced_s / span_s if span_s > 0 else 0.0
    if span_s > 0 and coverage < 30:
        problems.append(
            f'only {coverage:.0f}% of the {span_s:.0f}s it spans is inside '
            f'a note ({voiced_s:.0f}s over {len(rows)} notes) -- '
            'under-extracted; lower conf_min, or fmin if the singer goes '
            'low')
    elif span_s > 0 and coverage < 45:
        notices.append(f'{coverage:.0f}% voiced coverage is low')

    if not problems:
        OK_SONGS.append(stem)
    print(f'{stem[:30]:<30} {len(rows):5d} {ends:7.1f} {audio_dur:7.1f} '
          f'{first:5.1f} {onset if onset is not None else -1:6.1f} '
          f'{spread:4d} {same_pitch_runs:5d} {equal_share:5.0f}% '
          f'{med:6.2f}')
    for p in problems:
        print(f'{"":>30}   !! {p}')
    for n in notices:
        print(f'{"":>30}   -- {n}')

print('-' * 104)
print(f'{len(OK_SONGS)} of {len(RESULTS)} song(s) passed every check.')
print('These checks include two the repo validator cannot run, because')
print('they need the pitch track: whether a note sits outside the sung')
print('span, and where that span is. So a file can pass')
print('backend/check_ground_truth.py and still be flagged here.')
print('\nFor reference, the previous syllable-divided run measured 100%')
print('equal splits and ~50% of rows inside same-pitch runs. Both numbers')
print('should now be low.')

# ---- word-label health --------------------------------------------------
#
# Only meaningful when labels were asked for. They are debug-only: the app
# groups segments by the singer's own recognised words and by time, and
# GroundTruthService reads a missing `syllable` as empty. So nothing in
# this section says whether a ground truth will score correctly.
LABELLED = sorted(st for st, m in LABEL_SOURCE.items() if m != 'none')

if not LABELLED:
    print()
    print(f'Labels were off (LABEL_MODE={LABEL_MODE!r}), so every')
    print('`syllable` came out empty.')
    print()
    print('Nothing scores them. The app groups segments by the singer own')
    print('recognised words and by time, never by the ground truth text,')
    print('and GroundTruthService reads a missing syllable as empty.')
    print()
    print('Turn them on (set LABEL_MODE, then re-run Cells 5, 6 and 8) if')
    print('a file passes every check above and still sounds offset. A')
    print('constant shift that still fits inside the audio is invisible to')
    print('every check up there; reading the syllable under a note and')
    print('listening for it is the only way to see that one.')
else:
    print()
    print('=' * 104)
    print('WORD LABELS  (debug-only; nothing is scored against these)')
    print('=' * 104)
    print(f'{"song":<30} {"by":>8} {"notes":>6} {"blank":>6} {"uniq":>6} '
          f'{"top label":>18} {"share":>6}')
    print('-' * 104)
    for stem in LABELLED:
        rows = RESULTS[stem]
        labels = [r['syllable'] for r in rows]
        named = [t for t in labels if t]
        uniq = len(set(named))
        if named:
            top = max(set(named), key=named.count)
            share = 100.0 * named.count(top) / len(named)
        else:
            top, share = '--', 0.0
        print(f'{stem[:30]:<30} {LABEL_SOURCE[stem]:>8} {len(rows):6d} '
              f'{len(labels) - len(named):6d} {uniq:6d} '
              f'{top[:18]:>18} {share:5.0f}%')
    print('-' * 104)
    print('blank  = notes no label covered. Under forced alignment these')
    print('         are real: notes sung where the sheet has no word.')
    if LABEL_UNIT == 'syllable' and 'align' in LABEL_SOURCE.values():
        print('share  = how much of the song one SYLLABLE claims. A high')
        print('         number is not a fault here: `lahat` occurs 60')
        print('         times in Nasa Iyo Na Ang Lahat, so `hat` at 13%')
        print('         is arithmetic. The looping threshold this line')
        print('         used to carry was about Whisper, not alignment.')
    else:
        print('share  = how much of the song one word claims; past ~15% is')
        print('         Whisper looping, which sustained singing provokes')
    if 'whisper' in LABEL_SOURCE.values():
        print()
        print('Songs labelled `whisper` are the ones to doubt: free')
        print('transcription of this catalog measured 35% WER. Add their')
        print('lyrics to Cell 5 and they get forced-aligned instead.')


## Cell 10 — preview, export, download

In [ ]:
import json
import os
import shutil
from google.colab import files

os.makedirs(OUT_DIR, exist_ok=True)

for stem, rows in sorted(RESULTS.items()):
    flag = '' if stem in OK_SONGS else '   <-- FAILED A CHECK, see Cell 7'
    print(f'\n{stem}  ({len(rows)} notes){flag}')
    for r in rows[:10]:
        print(f"    {r['start']:7.2f} - {r['end']:7.2f}  "
              f"{r['note']:<4}  {r['syllable']}")

    # Exactly the shape ground_truth_service.dart parses: a top-level
    # "data" list of {syllable, start, end, note}, times in seconds.
    payload = {'data': [{'syllable': r['syllable'], 'start': r['start'],
                         'end': r['end'], 'note': r['note']}
                        for r in rows]}
    with open(os.path.join(OUT_DIR, f'{stem}.json'), 'w',
              encoding='utf-8') as fh:
        json.dump(payload, fh, ensure_ascii=False, indent=1)

    # Word onsets, when this song was forced-aligned. A separate file
    # because the app's own schema has no room for them, and because only
    # backend/sync_lyrics.py reads them.
    spans = WORD_SPANS.get(stem)
    if spans:
        words = {'stem': stem,
                 'source': 'MMS_FA forced alignment of the vocal stem',
                 'words': [{'text': w['text'],
                            'start': round(w['start'], 3),
                            'end': round(w['end'], 3),
                            'score': round(w.get('score', 0.0), 4)}
                           for w in spans]}
        with open(os.path.join(OUT_DIR, f'{stem}.words.json'), 'w',
                  encoding='utf-8') as fh:
            json.dump(words, fh, ensure_ascii=False, indent=1)
        print(f'    + {len(spans)} word onsets -> {stem}.words.json')

zip_base = '/content/ground_truth_notes'
shutil.make_archive(zip_base, 'zip', OUT_DIR)
print(f'\nWrote {len(RESULTS)} file(s). Downloading...')
files.download(zip_base + '.zip')

print('\nUnzip into assets/ground_truth/, keeping the exact filenames')
print('"<Title> - <Artist>.json" so they pair with assets/audio/.')
print('\nThen, locally:')
print('  python backend/check_ground_truth.py')
print('which re-runs these same checks against what you actually shipped.')
if any(WORD_SPANS.values()):
    print('')
    print('Word onsets were exported too. To re-time the scrolling')
    print('lyrics from them:')
    print('  python backend/sync_lyrics.py --all --write')